## Imports

In [8]:
import json
import base64
import boto3
from pathlib import Path
from collections import Counter

## Configuration

In [9]:
# SageMaker endpoint name — adjust to match your deployed endpoint
ENDPOINT_NAME = "hwi-adsv1-concurrency-20"
REGION = "us-west-2"

# Local paths
IMAGE_DIR = Path.home() / "Downloads" / "hawaii-validation-sample"
ADDAX_JSON = IMAGE_DIR / "image_recognition_file.json"

session = boto3.Session(profile_name="animl")
runtime = session.client("sagemaker-runtime", region_name=REGION)

## Load AddaxAI Results

In [10]:
with open(ADDAX_JSON) as f:
    addax_data = json.load(f)

cls_cats = addax_data.get("classification_categories", {})

det_count = sum(
    1 for img in addax_data["images"]
    for det in img.get("detections", [])
    if det.get("classifications")
)
print(f"Images: {len(addax_data['images'])}")
print(f"Detections with classifications: {det_count}")
print(f"Categories: {json.dumps(cls_cats, indent=2)}")

Images: 100
Detections with classifications: 101
Categories: {
  "1": "axis deer",
  "2": "bird",
  "3": "cat",
  "4": "cow",
  "5": "dog",
  "6": "donkey",
  "7": "false detection",
  "8": "goat",
  "9": "horse",
  "10": "mongoose",
  "11": "mule deer",
  "12": "rock wallaby",
  "13": "rodent",
  "14": "sheep",
  "15": "wild pig"
}


## Verify Endpoint

In [11]:
for img in addax_data["images"]:
    for det in img.get("detections", []):
        if not det.get("classifications"):
            continue
        with open(IMAGE_DIR / img["file"], "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()
        resp = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({"image": img_b64, "bbox": det["bbox"]}),
        )
        result = json.loads(resp["Body"].read())
        top = max(result, key=result.get)
        print(f"Endpoint is working: {img['file']} -> {top} ({result[top]:.4f})")
        break
    else:
        continue
    break

Endpoint is working: kauai_program_-005869a30facc0545e052d008c45e06b.jpg -> bird (0.8950)


## Run Full Distribution Comparison

Compares all 15 class probabilities per detection, not just the top prediction.

In [12]:
results = []

for img in addax_data["images"]:
    for det in img.get("detections", []):
        cls = det.get("classifications", [])
        if not cls:
            continue

        with open(IMAGE_DIR / img["file"], "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()

        resp = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({"image": img_b64, "bbox": det["bbox"]}),
        )
        preds = json.loads(resp["Body"].read())

        # Build AddaxAI full distribution from classifications list
        addax_scores = {}
        for cls_id, conf in cls:
            name = cls_cats.get(str(cls_id), str(cls_id))
            addax_scores[name] = conf

        # Compare all classes present in both
        all_classes = set(preds.keys()) | set(addax_scores.keys())
        max_diff = 0
        diffs = {}
        for c in all_classes:
            ours = preds.get(c, 0)
            theirs = addax_scores.get(c, 0)
            diff = abs(ours - theirs)
            if diff > 0.0001:
                diffs[c] = (ours, theirs, diff)
            max_diff = max(max_diff, diff)

        # Top-1 comparison
        our_top = max(preds, key=preds.get)
        addax_top_id, addax_top_conf = cls[0]
        addax_top = cls_cats.get(str(addax_top_id), str(addax_top_id))

        results.append({
            "file": img["file"],
            "top_match": our_top == addax_top,
            "max_diff": max_diff,
            "class_diffs": diffs,
            "addax_top": addax_top,
            "our_top": our_top,
            "addax_classes_checked": len(addax_scores),
            "our_classes_returned": len(preds),
        })

        if len(results) % 20 == 0:
            print(f"  ... processed {len(results)} detections")

print(f"Done. Processed {len(results)} detections.")

  ... processed 20 detections
  ... processed 40 detections
  ... processed 60 detections
  ... processed 80 detections
  ... processed 100 detections
Done. Processed 101 detections.


## Results

In [13]:
top_matches = sum(1 for r in results if r["top_match"])
exact_dist = sum(1 for r in results if not r["class_diffs"])
max_diff_overall = max(r["max_diff"] for r in results)

print(f"Top-1 class match: {top_matches}/{len(results)}")
print(f"Full distribution match (±0.0001): {exact_dist}/{len(results)}")
print(f"Max confidence difference across all classes and detections: {max_diff_overall:.6f}")
print()

# Species coverage
tested_species = Counter(r["addax_top"] for r in results)
all_species = set(cls_cats.values())
untested = all_species - set(tested_species.keys())
print(f"Species tested ({len(tested_species)}/{len(all_species)}):")
for name, count in tested_species.most_common():
    print(f"  {name}: {count}")
if untested:
    print(f"\nSpecies NOT tested: {', '.join(sorted(untested))}")
print()

# Show any mismatches
problems = [r for r in results if r["class_diffs"] or not r["top_match"]]
if problems:
    print(f"{len(problems)} detections with differences:")
    for r in problems:
        print(f"  ❌ {r['file']}")
        if not r["top_match"]:
            print(f"     Top-1 mismatch: AddaxAI={r['addax_top']}, Ours={r['our_top']}")
        for c, (ours, theirs, diff) in r["class_diffs"].items():
            print(f"     {c}: ours={ours:.6f}, addax={theirs:.6f}, diff={diff:.6f}")
        print()
else:
    print("✅ All predictions match exactly across all classes.")

Top-1 class match: 101/101
Full distribution match (±0.0001): 101/101
Max confidence difference across all classes and detections: 0.000007

Species tested (10/15):
  bird: 41
  mule deer: 33
  wild pig: 15
  sheep: 4
  axis deer: 2
  cow: 2
  cat: 1
  false detection: 1
  goat: 1
  rock wallaby: 1

Species NOT tested: dog, donkey, horse, mongoose, rodent

✅ All predictions match exactly across all classes.
